# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeerMusabih/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Unit of Analysis + Time Window

## Unit of Analysis

One row represents one content item for one client on one reporting date.

The warehouse grain is:

**report_date × client_hash_id × content_hash_id**

This notebook uses March 2026 as the development window. A middle-panel month is selected to avoid using the final month as development data.

The purpose of this data contract is to identify observable signals that can support ranking content pages by refresh opportunity.

## Verify Grain

In [37]:
query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM fact_content_daily_performance_sample
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

con.execute(query).df()

,report_date,client_hash_id,content_hash_id,row_count


## Verify Window

In [38]:
query = """
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT client_hash_id) AS clients
FROM fact_content_daily_performance_sample
"""

con.execute(query).df()

,first_date,last_date,total_rows,content_items,clients
0,2025-01-27,2025-01-31,1297,476,2


# 2. Fields: feature / label / context / excluded

## Features

The selected features represent information available before a refresh decision:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events


## Label / Proxy

This dataset does not contain a direct business label for "refresh opportunity".

A future model could use a proxy based on observed performance decline, but future outcome variables should not be used as features.


## Context

The following fields are used for identification and filtering:

- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4


IDs are context only and are not used as model features.


## Excluded

Excluded:

- client_hash_id
- content_hash_id

Reason:
These identify entities but do not represent generalizable performance signals.

Data availability flags are also used only for filtering because missing data represents unavailable tracking rather than zero activity.

# 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Query 1: Count for March 2026


In [39]:
query = """
SELECT
    month,
    COUNT(*) AS rows,
    COUNT(DISTINCT content_hash_id) AS content_items,
    COUNT(DISTINCT client_hash_id) AS clients
FROM fact_content_daily_performance_sample
WHERE month = '2026-03'
GROUP BY month
"""

con.execute(query).df()

,month,rows,content_items,clients


## Query 2: Availability Check

In [40]:
query = """
SELECT
    COUNT(*) AS available_gsc_rows
FROM fact_content_daily_performance_sample
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
"""

con.execute(query).df()

,available_gsc_rows
0,0


## Query 3: Missing Values

In [41]:
query = """
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END)
        AS impressions_missing,

    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END)
        AS clicks_missing,

    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END)
        AS position_missing,

    AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END)
        AS sessions_missing

FROM fact_content_daily_performance_sample
WHERE month = '2026-03'
"""

con.execute(query).df()

,impressions_missing,clicks_missing,position_missing,sessions_missing
0,NaN,NaN,NaN,NaN


## Five Feature Frame

In [42]:
query = """
SELECT
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events

FROM fact_content_daily_performance_sample

WHERE month = '2026-03'
AND gsc_data_available IS TRUE
"""

feature_frame = con.execute(query).df()

feature_frame.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events


## Feature Availability

### gsc_impressions

Available because search visibility has already been observed before a refresh decision.

### gsc_clicks

Available because historical clicks are recorded before deciding which pages require attention.

### gsc_avg_position

Available because ranking position is measured from existing search performance.

### ga4_sessions

Available because historical user sessions are observed before prioritization.

### scroll_events

Available because engagement behaviour has already occurred before the decision moment.

# 4. Data Limits

This dataset captures observed search and engagement behaviour but cannot directly measure content quality, relevance, competitor changes, or whether a refresh will cause improvement.

Some clients have incomplete tracking history. Missing values may represent unavailable measurement rather than zero activity.

The output should be treated as decision-support for prioritization, not causal evidence.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.